In [ ]:
#load data and imports

from sylegendarium import load_experiments
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
df_raw_orig = load_experiments("experiments")


In [ ]:
#information about model and initialization
print(df_raw_orig.columns)
results_df = df_raw_orig.copy()
print(results_df['run'].unique())
print(results_df['model_name'].unique())
print(results_df['policy_name'].unique())
#print mean rmse across models at last step
def _last_step_per_run(results_df):
    """Last logged step of each (run, policy, model)."""
    return (
        results_df.sort_values(["run", "policy_name", "model_name", "step"])
        .drop_duplicates(subset=["run", "policy_name", "model_name"], keep="last")
    )

last_df = _last_step_per_run(results_df.dropna(subset=["mse"]))

mean_rmse = results_df.groupby(['model_name'])['rmse'].mean().reset_index()

print(mean_rmse)
mean_iou = results_df.groupby(['model_name'])['iou'].mean().reset_index()
print(mean_iou)

In [ ]:
#useful functions for plotting and reporting
import ast
from pathlib import Path

import matplotlib

# ── Shared names, colours and IEEE style (used by every paper figure below) ──
MODEL_NAMES = {
    "gaussian_process": "GP",
    "mcdropout": "MC Dropout",
    "ensemble": "Ensemble",
    "edl": "EDL",
}
POLICY_NAMES = {
    "epsilon_greedy": r"$\epsilon$-greedy",
    "orienteering": "Receding Horizon Orienteering",
    "myopic_greedy": "Value Greedy",
    "mcts": "MCTS",
    "uncertainty_greedy": "Uncertainty Greedy",
}
# Same order and colours as the calibration figure of the paper
MODEL_ORDER = ["GP", "MC Dropout", "Ensemble", "EDL"]
MODEL_COLORS = dict(zip(MODEL_ORDER, sns.color_palette("Set2", len(MODEL_ORDER))))
MODEL_MARKERS = dict(zip(MODEL_ORDER, ["o", "s", "^", "D"]))
POLICY_ORDER = list(POLICY_NAMES)
POLICY_COLORS = dict(zip(POLICY_ORDER, sns.color_palette("tab10", len(POLICY_ORDER))))
POLICY_MARKERS = dict(zip(POLICY_ORDER, ["o", "s", "^", "D", "v"]))

IEEE_COLUMN_WIDTH = 3.5  # inches (88.9 mm)
IEEE_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,  # embed TrueType fonts (IEEE PDF eXpress)
    "ps.fonttype": 42,
}
# Shorter labels so legends fit in one IEEE column
IEEE_POLICY_LABELS = {**POLICY_NAMES, "orienteering": "RH Orienteering"}


def print_example_trajectory_report(results_df, experiment_id):
    example_df = results_df[results_df["run"] == experiment_id]
    rmse = example_df["mse"].values[-1]
    iou = example_df["iou"].values[-1]

    print(
        f"\nExample Trajectory Report for Experiment ID {experiment_id}:\n"
    )
    print(f"RMSE: {rmse:.4f}, IoU: {iou:.4f}")


def plot_example_trajectory(results_df, experiment_id):
    policy_mapping = {
        "epsilon_greedy": r"$\epsilon$-greedy",
        "orienteering": "Receding Horizon\nOrienteering",
        "myopic_greedy": "Value Greedy",
        "mcts": "MCTS",
        "uncertainty_greedy": "Uncertainty Greedy",
    }

    example_df = results_df.loc[
        results_df["run"] == experiment_id
    ].copy()

    example_df["Policy"] = example_df["policy_name"].replace(
        policy_mapping
    )

    policies = list(example_df["Policy"].unique())
    models = list(example_df["model_name"].unique())

    fig, axes = plt.subplots(
        len(policies),
        len(models),
        figsize=(4 * len(models), 4 * len(policies)),
        squeeze=False,
    )

    colors = ["r-", "g-", "b-", "c-"]

    for row, policy in enumerate(policies):
        for col, model in enumerate(models):
            ax = axes[row, col]

            policy_df = example_df[
                (example_df["Policy"] == policy)
                & (example_df["model_name"] == model)
            ]

            if policy_df.empty:
                ax.set_visible(False)
                continue

            trajectories = [[] for _ in range(4)]

            for position_value in policy_df["position"]:
                position = (
                    ast.literal_eval(position_value)
                    if isinstance(position_value, str)
                    else position_value
                )

                for trajectory_id, (y, x) in enumerate(position[:4]):
                    trajectories[trajectory_id].append([x, y])

            ground_truth = policy_df["ground_truth"].values[0]

            ax.imshow(
                ground_truth,
                cmap="viridis",
                interpolation="bicubic",
            )

            for color_id, trajectory in enumerate(trajectories):
                if trajectory:
                    trajectory = np.asarray(trajectory)

                    ax.plot(
                        trajectory[:, 0],
                        trajectory[:, 1],
                        colors[color_id],
                        marker=".",
                        alpha=0.5,
                    )

            ax.set_title(f"{policy}\n{model}")
            ax.set_aspect("equal")
            ax.set_xlabel("X")
            ax.set_ylabel("Y")
            ax.grid(False)

    plt.tight_layout()
    plt.show()

    return fig, axes

def _last_step_per_run(results_df):
    """Last logged step of each (run, policy, model)."""
    return (
        results_df.sort_values(["run", "policy_name", "model_name", "step"])
        .drop_duplicates(subset=["run", "policy_name", "model_name"], keep="last")
    )


def _ieee_boxplot(df, y, ylabel, filename, log_scale=False, headroom=0.12, output_dir=Path("."), height=2.0):
    """
    Boxplot (one group per model, one box per policy) sized for one IEEE column, saved as PDF + PNG.
    The legend goes in a single row at the top of the axes; `headroom` is the fraction of the
    axes height kept free for it above the data.
    """
    models = [m for m in MODEL_NAMES if m in set(df["model_name"])]
    policies = [p for p in POLICY_ORDER if p in set(df["policy_name"])]
    plot_df = df.assign(
        Model=df["model_name"].map(MODEL_NAMES),
        Policy=df["policy_name"].map(IEEE_POLICY_LABELS),
    )
    hue_order = [IEEE_POLICY_LABELS[p] for p in policies]

    # Layout in inches so the width is exactly one IEEE column
    left, right, top, bottom = 0.42, 0.04, 0.05, 0.2
    with plt.rc_context(IEEE_RC), sns.axes_style("darkgrid", {"font.family": "serif"}):
        fig, ax = plt.subplots(figsize=(IEEE_COLUMN_WIDTH, height))
        fig.subplots_adjust(left=left / IEEE_COLUMN_WIDTH, right=1 - right / IEEE_COLUMN_WIDTH,
                            bottom=bottom / height, top=1 - top / height)

        sns.boxplot(
            data=plot_df, x="Model", y=y, hue="Policy",
            order=[MODEL_NAMES[m] for m in models], hue_order=hue_order,
            palette={IEEE_POLICY_LABELS[p]: POLICY_COLORS[p] for p in policies},
            saturation=0.9, width=0.8, gap=0.1, linewidth=0.5, linecolor="0.25",
            flierprops={"marker": "o", "markersize": 1.8, "markerfacecolor": "0.35",
                        "markeredgewidth": 0, "alpha": 0.8},
            ax=ax,
        )

        if log_scale:
            ax.set_yscale("log")
        # Free space above the data for the legend
        to_axis, from_axis = (np.log10, lambda v: 10 ** v) if log_scale else (lambda v: v, lambda v: v)
        low, high = map(to_axis, ax.get_ylim())
        ax.set_ylim(from_axis(low), from_axis(low + (high - low) / (1 - headroom)))
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.tick_params(pad=1)
        ax.legend(title=None, loc="upper center", ncol=len(hue_order), frameon=False, fontsize=6,
                  handlelength=0.9, handleheight=0.6, columnspacing=0.7, handletextpad=0.3,
                  borderaxespad=0.25)

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        for ext in ("pdf", "png"):
            fig.savefig(output_dir / f"{filename}.{ext}", dpi=300)
        print(f"Saved: {output_dir / filename}.pdf / .png")
    plt.show()
    return fig, ax


def plot_boxplot_models(results_df, output_dir=Path("."), log_scale=True):
    """MSE of the predicted map at the end of each run, across models and policies."""
    df = results_df.copy()
    df["mse"] = pd.to_numeric(df["mse"], errors="coerce")
    df = _last_step_per_run(df.dropna(subset=["mse"]))

    return _ieee_boxplot(df, "mse", "MSE", "mse_boxplot", log_scale=log_scale, output_dir=output_dir)


def plot_boxplot_iou(results_df, output_dir=Path(".")):
    """IoU of the oil-spill mask at the end of each run, across models and policies."""
    df = results_df.copy()
    df["iou"] = pd.to_numeric(df["iou"], errors="coerce")
    df = _last_step_per_run(df.dropna(subset=["iou"]))

    return _ieee_boxplot(df, "iou", "IoU", "iou_boxplot", output_dir=output_dir)


def plot_map_evolution(
    results_df,
    run_id,
    map_column,
    max_columns=5,
    cmap="viridis",
    figsize=None,
):
    """Plot a spatial map at each step of one run."""
    import ast

    required_columns = {"run", "step", map_column, "model_name", "policy_name"}
    missing_columns = required_columns.difference(results_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    if max_columns < 1:
        raise ValueError("max_columns must be at least 1")

    run_df = results_df.loc[results_df["run"] == run_id].sort_values("step")
    if run_df.empty:
        raise ValueError(f"No rows found for run_id={run_id}")

    maps = []
    steps = []
    identificators = []
    for _, row in run_df.iterrows():
        current_map = row[map_column]
        if isinstance(current_map, str):
            current_map = ast.literal_eval(current_map)
        current_map = np.asarray(current_map)
        if current_map.ndim != 2:
            raise ValueError(
                f"{map_column} at step {row['step']} must be 2D, "
                f"got shape {current_map.shape}"
            )
        maps.append(current_map)
        steps.append(row["step"])
        identificators.append(f"{row['model_name']}_{row['policy_name']}")

    n_panels = len(maps)
    n_columns = min(max_columns, n_panels)
    n_rows = int(np.ceil(n_panels / n_columns))
    if figsize is None:
        figsize = (3.2 * n_columns, 3.2 * n_rows)

    fig, axes = plt.subplots(n_rows, n_columns, figsize=figsize, squeeze=False)
    axes = axes.ravel()
    vmin = min(current_map.min() for current_map in maps)
    vmax = max(current_map.max() for current_map in maps)

    for ax, step, current_map, identificator in zip(axes, steps, maps, identificators):
        image = ax.imshow(current_map, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"step {step}_{identificator}")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    for ax in axes[n_panels:]:
        ax.remove()

    # fig.suptitle(f"Evolution of {map_column} - run {run_id}")
    fig.tight_layout()
    return fig, axes[:n_panels]


def plot_predicted_mean_evolution(results_df, run_id, **kwargs):
    return plot_map_evolution(
        results_df, run_id, map_column="predicted_mean", **kwargs
    )


def plot_predicted_uncertainty_evolution(results_df, run_id, **kwargs):
    return plot_map_evolution(
        results_df, run_id, map_column="predicted_uncertainty", **kwargs
    )

def plot_obs_map_evolution(results_df, run_id, **kwargs):
    return plot_map_evolution(
        results_df, run_id, map_column="obs_map", **kwargs
    )


# Examples:
# plot_predicted_mean_evolution(results_df, run_id=0)
# plot_predicted_uncertainty_evolution(results_df, run_id=0)
# plt.show()



def heat_map_RMSE_policy_model(results_df):
    """
    Plots a heatmap comparing the RMSE of different policies and models.

    Parameters:
    results_df (pd.DataFrame): A DataFrame containing the results of different policies and models.
    """
    
    df = results_df.copy()

    df["RMSE_normalized"] = df.groupby(["run", "policy_name"])["mse"].transform(lambda x: x / x.iloc[0])  # Normalize RMSE by distance
    
    # Select the last entry for each experiment_id and policy and model to avoid duplicates
    df = df.drop_duplicates(subset=['run', 'policy_name', 'model_name'], keep='last')
    
    df = df.rename(columns={"policy_name": "Policy", "distance_bin": "Distance (px)", "RMSE_normalized": "Normalized RMSE", "model_name": "Model"})

    
    policy_mapping = {
        "epsilon_greedy": r"$\epsilon$-greedy",
        "orienteering": "Receeding Horizon Orienteering",
        "myopic_greedy": "Value Greedy",
        "mcts": "MCTS",
        "uncertainty_greedy": "Uncertainty Greedy",
    }
    df.loc[:, "Policy"] = df["Policy"].replace(policy_mapping)
    
    model_mapping = {
        "ensemble": "Deep Ensemble",
        "gaussian_process": "Gaussian Process",
    }
    df.loc[:, "Model"] = df["Model"].replace(model_mapping)
     
    pivot_table = df.pivot_table(index='Model', columns='Policy', values='Normalized RMSE', aggfunc='mean')
    
    # Order by the average RMSE across policies for better visualization
    pivot_table = pivot_table.loc[pivot_table.mean(axis=1).sort_values().index]
    
    plt.figure(figsize=(8, 6))
    
    sns.heatmap(pivot_table, annot=True, fmt=".2f", cmap="YlGnBu")
    
    plt.title("Heatmap of RMSE by Policy and Model")
    
    plt.tight_layout()
    
    # Show the plot
    # plt.show()

    # plt.savefig(ROOT / "heatmap_rmse_policy_model.png", dpi=300)  # Save the figure with high resolution
    # # Save also as svg for better quality in publications
    # plt.savefig(ROOT / "heatmap_rmse_policy_model.svg")

In [ ]:
#plot trajectories and reports for each experiment
for example_experiment_id in range(len(results_df['run'].unique())):
        plot_example_trajectory(results_df, example_experiment_id)
        print_example_trajectory_report(results_df, example_experiment_id)

In [ ]:
#plot all maps (evolution, mean and uncertainty during a simulation run)
model_df = results_df[results_df["model_name"] == "edl"]

fig, axes = plot_obs_map_evolution(model_df, run_id=0)
fig, axes = plot_predicted_mean_evolution(model_df, run_id=0)
fig, axes = plot_predicted_uncertainty_evolution(model_df, run_id=0)
plt.show()

In [ ]:
#plot heatmap of RMSE for each policy and model
heat_map_RMSE_policy_model(results_df)

In [ ]:
#boxplot comparison of models
plot_boxplot_models(results_df)

In [ ]:
plot_boxplot_iou(results_df)

In [ ]:
def plot_predicted_mean_vs_ground_truth_by_run_and_model_policy(df):
    """
    For each model-policy pair, plot one row per run with:
    left = ground truth
    right = predicted mean
    """
    import ast

    required_columns = {
        "run",
        "step",
        "model_name",
        "policy_name",
        "predicted_mean",
        "ground_truth",
    }
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    df = df.sort_values(["model_name", "policy_name", "run", "step"]).copy()

    for (model_name, policy_name), group_df in df.groupby(["model_name", "policy_name"], sort=True):
        run_ids = sorted(group_df["run"].dropna().unique())
        if not len(run_ids):
            continue

        fig, axes = plt.subplots(
            len(run_ids),
            2,
            figsize=(12, 3.5 * len(run_ids)),
            squeeze=False,
        )

        for row_idx, run_id in enumerate(run_ids):
            run_df = group_df[group_df["run"] == run_id].sort_values("step")
            if run_df.empty:
                continue

            last_row = run_df.iloc[-1]
            gt_map = last_row["ground_truth"]
            pred_map = last_row["predicted_mean"]

            if isinstance(gt_map, str):
                gt_map = ast.literal_eval(gt_map)
            if isinstance(pred_map, str):
                pred_map = ast.literal_eval(pred_map)

            gt_map = np.asarray(gt_map, dtype=float)
            pred_map = np.asarray(pred_map, dtype=float)

            if gt_map.shape != pred_map.shape:
                raise ValueError(
                    f"Shape mismatch for run {run_id} ({model_name}, {policy_name}): "
                    f"ground_truth {gt_map.shape}, predicted_mean {pred_map.shape}"
                )

            gt_mask = gt_map > 0.1
            pred_mask = pred_map > 0.1

            ax_gt = axes[row_idx, 0]
            ax_pred = axes[row_idx, 1]

            ax_gt.imshow(gt_map, cmap="Greys", alpha=0.7)
            ax_gt.contour(gt_mask.astype(int), levels=[0.5], colors="dodgerblue", linewidths=2)
            ax_gt.set_title(f"Ground truth\nRun {run_id}")
            ax_gt.set_aspect("equal")
            ax_gt.set_xlabel("X")
            ax_gt.set_ylabel("Y")

            ax_pred.imshow(pred_map, cmap="Greys", alpha=0.7)
            ax_pred.contour(pred_mask.astype(int), levels=[0.5], colors="firebrick", linewidths=2)
            ax_pred.set_title(f"Predicted mean\nRun {run_id}")
            ax_pred.set_aspect("equal")
            ax_pred.set_xlabel("X")
            ax_pred.set_ylabel("Y")

        fig.suptitle(
            f"Model: {model_name} | Policy: {policy_name}",
            y=1.02,
            fontsize=12,
        )
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

plot_predicted_mean_vs_ground_truth_by_run_and_model_policy(results_df)

In [ ]:
#mean time of each model
def mean_time_per_model(results_df):
    """
    Computes the mean time taken for each model across all runs.

    Parameters:
    results_df (pd.DataFrame): A DataFrame containing the results of different policies and models.

    Returns:
    pd.DataFrame: A DataFrame with the mean time for each model.
    """
    if 'model_mean_time' not in results_df.columns:
        raise ValueError("The 'model_mean_time' column is missing from the DataFrame.")

    mean_time_df = results_df.groupby('model_name')['model_mean_time'].mean().reset_index()
    mean_time_df.rename(columns={'model_mean_time': 'Mean Time'}, inplace=True)
    
    return mean_time_df

def mean_time_per_policy(results_df):
    """
    Computes the mean time taken for each policy across all runs.

    Parameters:
    results_df (pd.DataFrame): A DataFrame containing the results of different policies and models.

    Returns:
    pd.DataFrame: A DataFrame with the mean time for each policy.
    """
    if 'policy_mean_time' not in results_df.columns:
        raise ValueError("The 'policy_mean_time' column is missing from the DataFrame.")

    mean_time_df = results_df.groupby('policy_name')['policy_mean_time'].mean().reset_index()
    mean_time_df.rename(columns={'policy_mean_time': 'Mean Time'}, inplace=True)
    
    return mean_time_df
mean_time_df = mean_time_per_model(results_df)
print("\nMean Time per Model:")
#print with high precission in scientific notation
print(mean_time_df.to_string(index=False, float_format=lambda x: f"{x:.6e}"))

mean_time_df = mean_time_per_policy(results_df)
print("\nMean Time per Policy:")
print(mean_time_df)

In [ ]:
print(results_df['model_mean_time'])

In [ ]:
def print_predicted_mean_against_groundtruth(df):
    """
    Plots the predicted mean against the ground truth for each model and run.

    Parameters:
    df (pd.DataFrame): A DataFrame containing the results of different runs.
    """
    import ast

    required_columns = {"run", "predicted_mean", "ground_truth", "model_name"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    for model_name, model_df in df.groupby("model_name", sort=True):
        run_ids = sorted(model_df["run"].dropna().unique())
        if not run_ids:
            continue

        fig, axes = plt.subplots(
            len(run_ids),
            1,
            figsize=(12, 3.5 * len(run_ids)),
            squeeze=False,
        )

        for ax, run_id in zip(axes.flatten(), run_ids):
            run_df = model_df[model_df["run"] == run_id].sort_values("step")
            if run_df.empty:
                print(f"No data found for run_id={run_id}, model={model_name}")
                continue

            predicted_means = []
            ground_truths = []

            for _, row in run_df.iterrows():
                predicted_mean = row["predicted_mean"]
                ground_truth = row["ground_truth"]

                if isinstance(predicted_mean, str):
                    predicted_mean = ast.literal_eval(predicted_mean)
                if isinstance(ground_truth, str):
                    ground_truth = ast.literal_eval(ground_truth)

                predicted_means.append(np.asarray(predicted_mean).ravel())
                ground_truths.append(np.asarray(ground_truth).ravel())

            if predicted_means:
                predicted_means = np.concatenate(predicted_means)
                ground_truths = np.concatenate(ground_truths)
            else:
                predicted_means = np.array([])
                ground_truths = np.array([])

            ax.plot(predicted_means, label="Predicted Mean", alpha=0.7)
            ax.plot(ground_truths, label="Ground Truth", alpha=0.7)
            ax.set_title(f"Model: {model_name} - Run {run_id}")
            ax.set_xlabel("Index")
            ax.set_ylabel("Value")
            ax.grid(True, alpha=0.3)
            ax.legend()

        fig.suptitle(f"Predicted Mean vs Ground Truth - {model_name}", y=1.02)
        fig.tight_layout()
        plt.show()

print_predicted_mean_against_groundtruth(results_df)

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_last_run_uncertainty_by_policy(df):
    """
    One subplot per policy for the last run.
    X = mean uncertainty of the model for that run/policy
    Y = uncertainty value of each point
    One line per model
    """
    required = {"run", "model_name", "policy_name", "predicted_uncertainty"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas: {sorted(missing)}")

    df = df.copy()
    df["run"] = pd.to_numeric(df["run"], errors="coerce")
    df = df.dropna(subset=["run"]).copy()

    last_run = int(df["run"].max())
    df_last = df[df["run"] == last_run].copy()

    rows = []
    for _, row in df_last.iterrows():
        arr = row["predicted_uncertainty"]
        if isinstance(arr, str):
            arr = ast.literal_eval(arr)
        arr = np.asarray(arr, dtype=float).ravel()

        if arr.size == 0:
            continue

        mean_unc = float(arr.mean())

        for i, v in enumerate(arr):
            rows.append({
                "model_name": row["model_name"],
                "policy_name": row["policy_name"],
                "run": last_run,
                "mean_uncertainty": mean_unc,
                "point_uncertainty": float(v),
                "point_index": i
            })

    if not rows:
        raise ValueError(f"No hay datos válidos para el run {last_run}")

    plot_df = pd.DataFrame(rows)

    policies = sorted(plot_df["policy_name"].dropna().unique().tolist())
    models = sorted(plot_df["model_name"].dropna().unique().tolist())

    n_policies = len(policies)
    fig, axes = plt.subplots(1, n_policies, figsize=(5 * n_policies, 4), sharey=True)
    if n_policies == 1:
        axes = [axes]

    for ax, policy_name in zip(axes, policies):
        ax.set_title(f"Policy: {policy_name}")
        for model_name in models:
            sub = plot_df[
                (plot_df["policy_name"] == policy_name) &
                (plot_df["model_name"] == model_name)
            ].sort_values("point_index")

            if sub.empty:
                continue

            x = sub["mean_uncertainty"].to_numpy()
            y = sub["point_uncertainty"].to_numpy()
            ax.plot(x, y, marker="o", linewidth=2, label=model_name)

        ax.set_xlabel("Mean uncertainty")
        ax.set_ylabel("Uncertainty per point" if policy_name == policies[0] else "")
        ax.grid(True, alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Model", loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.suptitle(f"Último run: {last_run}", y=1.02)
    plt.tight_layout()
    plt.show()

plot_last_run_uncertainty_by_policy(results_df)

In [ ]:
#prediction error (predicted mean vs ground truth) and error-uncertainty calibration curve
# Uses MODEL_NAMES, POLICY_NAMES, MODEL_ORDER, MODEL_COLORS and MODEL_MARKERS from the utilities cell.


def _minmax(arr):
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)


def _error_and_sigma(row, border=8):
    """
    |predicted_mean - ground_truth| and sigma at unobserved cells, excluding the map border.
    Non-finite cells are dropped: some policies write -inf into the logged obs maps in place.
    """
    abs_err = np.abs(np.asarray(row["predicted_mean"], dtype=float) - np.asarray(row["ground_truth"], dtype=float))
    sigma = np.asarray(row["predicted_uncertainty"], dtype=float)
    eval_mask = (np.asarray(row["obs_mask"]) == 0) & np.isfinite(abs_err) & np.isfinite(sigma)
    if border:
        eval_mask[:border, :] = eval_mask[-border:, :] = False
        eval_mask[:, :border] = eval_mask[:, -border:] = False
    return abs_err[eval_mask], sigma[eval_mask]


def prediction_error_table(results_df, border=8):
    """MAE / RMSE of the predicted mean at unobserved cells, on the last step of each run."""
    df = results_df.sort_values("step").drop_duplicates(["run", "policy_name", "model_name"], keep="last")
    rows = []
    for _, row in df.iterrows():
        abs_err, _ = _error_and_sigma(row, border)
        if abs_err.size == 0:
            continue
        rows.append({
            "Model": MODEL_NAMES.get(row["model_name"], row["model_name"]),
            "Policy": POLICY_NAMES.get(row["policy_name"], row["policy_name"]),
            "MAE": abs_err.mean(),
            "RMSE": np.sqrt((abs_err ** 2).mean()),
        })
    table = pd.DataFrame(rows).groupby(["Model", "Policy"])[["MAE", "RMSE"]].agg(["mean", "std"])
    return table


def plot_error_maps(results_df, run_id, policy_name, step=None, border=8):
    """GT | predicted mean | |error| | sigma for each model, for one run and policy (last step by default)."""
    df = results_df[(results_df["run"] == run_id) & (results_df["policy_name"] == policy_name)]
    models = [m for m in MODEL_NAMES if m in set(df["model_name"])]
    fig, axes = plt.subplots(len(models), 4, figsize=(13, 3.2 * len(models)), squeeze=False)
    for ax_row, model in zip(axes, models):
        model_df = df[df["model_name"] == model].sort_values("step")
        row = model_df.iloc[-1] if step is None else model_df[model_df["step"] == step].iloc[0]
        gt = np.asarray(row["ground_truth"], dtype=float)
        mean = np.asarray(row["predicted_mean"], dtype=float)
        panels = [
            ("Ground truth", gt, "viridis", 0, 1),
            ("Predicted mean", mean, "viridis", 0, 1),
            ("|mean − GT|", np.abs(mean - gt), "magma", 0, None),
            ("σ (predicted uncertainty)", np.asarray(row["predicted_uncertainty"], dtype=float), "magma", 0, None),
        ]
        for ax, (title, image, cmap, vmin, vmax) in zip(ax_row, panels):
            im = ax.imshow(np.where(np.isfinite(image), image, np.nan), cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(f"{MODEL_NAMES[model]} — {title}", fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f"Run {run_id} | {POLICY_NAMES.get(policy_name, policy_name)} | step {row['step']}")
    fig.tight_layout()
    return fig, axes


def calibration_curves(results_df, n_bins=10, border=8, last_step_only=False):
    """
    Per-sample error-uncertainty calibration curves (same method as scenario/models/plot_results.py).

    For every prediction (one step of one run), sigma and |mean - GT| at unobserved cells are
    min-max normalised per sample and binned by percentiles of sigma; the bin means are
    normalised to [0, 1] and interpolated on a common grid.
    Returns (sigma_grid, DataFrame with one row per sample and its curve).
    """
    df = results_df.sort_values("step")
    if last_step_only:
        df = df.drop_duplicates(["run", "policy_name", "model_name"], keep="last")
    sigma_grid = np.linspace(0, 1, n_bins)

    curves = []
    for _, row in df.iterrows():
        abs_err, sigma = _error_and_sigma(row, border)
        if abs_err.size < n_bins or np.ptp(sigma) < 1e-8 or np.ptp(abs_err) < 1e-8:
            continue  # constant map (e.g. step 0), not informative
        sigma_n, err_n = _minmax(sigma), _minmax(abs_err)

        bin_edges = np.percentile(sigma_n, np.linspace(0, 100, n_bins + 1))
        bin_edges[-1] += 1e-8
        bin_idx = np.clip(np.digitize(sigma_n, bin_edges) - 1, 0, n_bins - 1)
        counts = np.bincount(bin_idx, minlength=n_bins)
        valid = counts > 0
        if valid.sum() < 2:
            continue
        bin_sigma = np.bincount(bin_idx, weights=sigma_n, minlength=n_bins)[valid] / counts[valid]
        bin_err = np.bincount(bin_idx, weights=err_n, minlength=n_bins)[valid] / counts[valid]

        curves.append({
            "model_name": row["model_name"],
            "policy_name": row["policy_name"],
            "dataset_name": row["dataset_name"],
            "run": row["run"],
            "step": row["step"],
            "curve": np.interp(sigma_grid, _minmax(bin_sigma), _minmax(bin_err)),
        })
    return sigma_grid, pd.DataFrame(curves)


def _plot_calibration_panels(panels, line_col, lines, colors, markers, sigma_grid, max_columns=3):
    """
    Draw one calibration subplot per (title, curves) panel and one line per (key, label) in
    `lines` (matched against `line_col`); each line is the mean over samples ± standard error.
    Returns (fig, axes, deviation table: mean |curve − ideal| per panel and line).
    """
    n_columns = min(max_columns, len(panels))
    n_rows = int(np.ceil(len(panels) / n_columns))
    with sns.axes_style("darkgrid"):
        fig, axes = plt.subplots(n_rows, n_columns, figsize=(5 * n_columns, 4.5 * n_rows), squeeze=False)
    axes = axes.ravel()

    deviation_rows = []
    for panel_idx, (ax, (title, panel_df)) in enumerate(zip(axes, panels)):
        ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, alpha=0.6, label="Ideal (y = x)", zorder=1)
        ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.05, color="tomato", label="Overconfident (error > σ)")
        ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.05, color="steelblue", label="Underconfident (error < σ)")

        for key, label in lines:
            line_curves = panel_df.loc[panel_df[line_col] == key, "curve"]
            if line_curves.empty:
                continue
            line_curves = np.stack(line_curves.to_list())
            mean_err = line_curves.mean(axis=0)
            se_err = line_curves.std(axis=0) / np.sqrt(len(line_curves))

            ax.plot(sigma_grid, mean_err, color=colors[key], linewidth=2.0, marker=markers[key],
                    markersize=5, label=label, zorder=3)
            ax.fill_between(sigma_grid, mean_err - se_err, mean_err + se_err, color=colors[key], alpha=0.2, zorder=2)
            deviation_rows.append({
                "Panel": title,
                "Line": label,
                "Mean |curve − ideal|": np.abs(mean_err - sigma_grid).mean(),
            })

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title)
        ax.set_xlabel("σ̃  (normalized uncertainty per sample)")
        if panel_idx % n_columns == 0:
            ax.set_ylabel("Mean absolute error (normalized per sample)")

    for ax in axes[len(panels):]:
        ax.remove()

    # Shared legend with every line that appears in any panel
    legend = {}
    for ax in axes[:len(panels)]:
        for handle, label in zip(*ax.get_legend_handles_labels()):
            legend.setdefault(label, handle)
    fig.legend(legend.values(), legend.keys(), loc="lower center", ncol=len(legend), fontsize=9, frameon=True,
               bbox_to_anchor=(0.5, -0.08 / n_rows))
    fig.tight_layout()

    deviation_df = pd.DataFrame(deviation_rows).pivot(index="Panel", columns="Line", values="Mean |curve − ideal|")
    deviation_df = deviation_df.loc[[title for title, _ in panels],
                                    [label for _, label in lines if label in deviation_df.columns]]
    deviation_df.columns.name = None
    return fig, axes[:len(panels)], deviation_df


def plot_error_uncertainty_calibration(results_df, facet_col="policy_name", n_bins=10, border=8,
                                       last_step_only=False, add_all_panel=True, max_columns=3):
    """
    Error-uncertainty calibration curves (see calibration_curves): one subplot per value of
    `facet_col`, one line per model.
    """
    sigma_grid, curves = calibration_curves(results_df, n_bins, border, last_step_only)

    facets = [f for f in POLICY_NAMES if f in set(curves[facet_col])] if facet_col == "policy_name"         else sorted(curves[facet_col].unique())
    panels = [(POLICY_NAMES.get(f, f), curves[curves[facet_col] == f]) for f in facets]
    if add_all_panel and len(facets) > 1:
        panels.append(("All", curves))

    lines = [(m, MODEL_NAMES[m]) for m in MODEL_NAMES if m in set(curves["model_name"])]
    colors = {m: MODEL_COLORS[MODEL_NAMES[m]] for m, _ in lines}
    markers = {m: MODEL_MARKERS[MODEL_NAMES[m]] for m, _ in lines}
    return _plot_calibration_panels(panels, "model_name", lines, colors, markers, sigma_grid, max_columns)


print("Prediction error at unobserved cells (last step of each run, mean ± std over runs):")
display(prediction_error_table(results_df).round(4))

plot_error_maps(results_df, run_id=0, policy_name="mcts")
plt.show()

fig, axes, calibration_deviation = plot_error_uncertainty_calibration(results_df)
plt.show()
print("Calibration: mean absolute deviation of each curve from the ideal diagonal (lower is better):")
display(calibration_deviation.round(4))

In [ ]:
#UCE (mean ± std over runs) for each policy and model
# Uses MODEL_NAMES, POLICY_NAMES and MODEL_ORDER from the utilities cell.
import sys
sys.path.append("scenario/models")
from metrics import uce


def uce_table(results_df, border=8, n_bins=10, last_step_only=True):
    """
    Uncertainty Calibration Error (scenario/models/metrics.py) of every prediction at
    unobserved cells, excluding the map border and non-finite cells.
    By default one value per run (its last step); mean ± std over runs.
    """
    df = results_df.sort_values("step")
    if last_step_only:
        df = df.drop_duplicates(["run", "policy_name", "model_name"], keep="last")

    rows = []
    for _, row in df.iterrows():
        gt = np.asarray(row["ground_truth"], dtype=float)
        mean = np.asarray(row["predicted_mean"], dtype=float)
        sigma = np.asarray(row["predicted_uncertainty"], dtype=float)
        eval_mask = (np.asarray(row["obs_mask"]) == 0) & np.isfinite(mean) & np.isfinite(sigma)
        if border:
            eval_mask[:border, :] = eval_mask[-border:, :] = False
            eval_mask[:, :border] = eval_mask[:, -border:] = False
        if not eval_mask.any():
            continue
        rows.append({
            "Model": MODEL_NAMES.get(row["model_name"], row["model_name"]),
            "Policy": POLICY_NAMES.get(row["policy_name"], row["policy_name"]),
            "UCE": uce(gt, mean, sigma, n_bins=n_bins, mask=eval_mask),
        })

    stats = pd.DataFrame(rows).groupby(["Model", "Policy"])["UCE"].agg(["mean", "std", "count"])
    table = (stats["mean"].map("{:.4f}".format) + " ± " + stats["std"].map("{:.4f}".format)).unstack("Policy")
    models = [m for m in MODEL_ORDER if m in table.index]
    policies = [p for p in POLICY_NAMES.values() if p in table.columns]
    return table.loc[models, policies], stats


uce_formatted, uce_stats = uce_table(results_df)
print("UCE at unobserved cells (last step of each run, mean ± std over runs, lower is better):")
display(uce_formatted)

In [ ]:
#ENCE (mean ± std over runs) for each policy and model
# Uses MODEL_NAMES, POLICY_NAMES and MODEL_ORDER from the utilities cell.


def ence_levi(y_true, y_pred_mean, y_pred_std, n_bins=10):
    """
    Expected Normalized Calibration Error (Levi et al., 2022, "Evaluating and Calibrating
    Uncertainty Prediction in Regression Tasks").

    Samples are sorted by sigma and split into n_bins bins with the same number of samples:

        RMV_j  = sqrt(mean(sigma² in B_j))
        RMSE_j = sqrt(mean((y − mu)² in B_j))
        ENCE   = (1/B) Σ_j |RMV_j − RMSE_j| / RMV_j

    A perfectly calibrated model (RMSE = sigma in every bin) yields ENCE = 0.
    Also returns the coefficient of variation C_v of sigma: ENCE is only meaningful
    together with some spread of sigma (a constant sigma can still get ENCE ≈ 0).
    """
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred_mean = np.asarray(y_pred_mean, dtype=float).ravel()
    sigma = np.maximum(np.asarray(y_pred_std, dtype=float).ravel(), 1e-6)
    sq_error = (y_true - y_pred_mean) ** 2

    terms = []
    for idx in np.array_split(np.argsort(sigma), n_bins):
        if idx.size == 0:
            continue
        rmv = np.sqrt(np.mean(sigma[idx] ** 2))
        rmse_bin = np.sqrt(np.mean(sq_error[idx]))
        terms.append(abs(rmv - rmse_bin) / rmv)

    c_v = np.std(sigma, ddof=1) / np.mean(sigma) if sigma.size > 1 else np.nan
    return float(np.mean(terms)), float(c_v)


def ence_table(results_df, border=8, n_bins=10, last_step_only=True):
    """
    ENCE of every prediction at unobserved cells, excluding the map border and non-finite cells.
    By default one value per run (its last step); mean ± std over runs.
    """
    df = results_df.sort_values("step")
    if last_step_only:
        df = df.drop_duplicates(["run", "policy_name", "model_name"], keep="last")

    rows = []
    for _, row in df.iterrows():
        gt = np.asarray(row["ground_truth"], dtype=float)
        mean = np.asarray(row["predicted_mean"], dtype=float)
        sigma = np.asarray(row["predicted_uncertainty"], dtype=float)
        eval_mask = (np.asarray(row["obs_mask"]) == 0) & np.isfinite(mean) & np.isfinite(sigma)
        if border:
            eval_mask[:border, :] = eval_mask[-border:, :] = False
            eval_mask[:, :border] = eval_mask[:, -border:] = False
        if eval_mask.sum() < n_bins:
            continue
        ence_value, c_v = ence_levi(gt[eval_mask], mean[eval_mask], sigma[eval_mask], n_bins=n_bins)
        rows.append({
            "Model": MODEL_NAMES.get(row["model_name"], row["model_name"]),
            "Policy": POLICY_NAMES.get(row["policy_name"], row["policy_name"]),
            "ENCE": ence_value,
            "C_v": c_v,
        })

    stats = pd.DataFrame(rows).groupby(["Model", "Policy"])[["ENCE", "C_v"]].agg(["mean", "std", "count"])
    models = [m for m in MODEL_ORDER if m in stats.index.get_level_values("Model")]
    policies = [p for p in POLICY_NAMES.values() if p in stats.index.get_level_values("Policy")]

    def _formatted(metric):
        table = (stats[(metric, "mean")].map("{:.4f}".format) + " ± "
                 + stats[(metric, "std")].map("{:.4f}".format)).unstack("Policy")
        return table.loc[models, policies]

    return _formatted("ENCE"), _formatted("C_v"), stats


ence_formatted, cv_formatted, ence_stats = ence_table(results_df)
print("ENCE at unobserved cells (last step of each run, mean ± std over runs, lower is better):")
display(ence_formatted)
print("Coefficient of variation of σ, C_v (higher = more spread in σ; read together with ENCE):")
display(cv_formatted)

In [ ]:
#error-uncertainty calibration curves: one subplot per model, one line per policy
# Uses calibration_curves and _plot_calibration_panels from the calibration cell above,
# and POLICY_ORDER / POLICY_COLORS / POLICY_MARKERS from the utilities cell.


def plot_calibration_by_model(results_df, n_bins=10, border=8, last_step_only=False, max_columns=4):
    """Calibration curves (see calibration_curves): one subplot per model, one line per policy."""
    sigma_grid, curves = calibration_curves(results_df, n_bins, border, last_step_only)

    models = [m for m in MODEL_NAMES if m in set(curves["model_name"])]
    panels = [(MODEL_NAMES[m], curves[curves["model_name"] == m]) for m in models]
    lines = [(p, POLICY_NAMES[p]) for p in POLICY_ORDER if p in set(curves["policy_name"])]
    return _plot_calibration_panels(panels, "policy_name", lines, POLICY_COLORS, POLICY_MARKERS, sigma_grid,
                                    max_columns)


fig, axes, calibration_deviation_by_model = plot_calibration_by_model(results_df)
plt.show()
print("Calibration: mean absolute deviation of each curve from the ideal diagonal (lower is better):")
display(calibration_deviation_by_model.round(4))

In [ ]:
#calibration curves by model sized for one IEEE column (3.5 in), saved as PDF + PNG
# Uses calibration_curves (calibration cell) and the shared names, colours and IEEE_RC
# from the utilities cell.


def plot_calibration_by_model_ieee(results_df, n_bins=10, border=8, last_step_only=False,
                                   output_dir=Path("Results/plots"), filename="calibration_by_model_ieee"):
    """Calibration curves (one subplot per model, one line per policy) in a 2×2 grid, one IEEE column wide."""
    sigma_grid, curves = calibration_curves(results_df, n_bins, border, last_step_only)
    models = [m for m in MODEL_NAMES if m in set(curves["model_name"])]
    policies = [p for p in POLICY_ORDER if p in set(curves["policy_name"])]

    # Layout in inches: square panels, fixed margins so the width is exactly one IEEE column
    n_columns = 2
    n_rows = int(np.ceil(len(models) / n_columns))
    left, right, col_gap = 0.42, 0.06, 0.10
    top, row_gap, bottom = 0.17, 0.20, 0.72
    panel = (IEEE_COLUMN_WIDTH - left - right - col_gap * (n_columns - 1)) / n_columns
    height = top + bottom + n_rows * panel + (n_rows - 1) * row_gap

    with plt.rc_context(IEEE_RC), sns.axes_style("darkgrid", {"font.family": "serif"}):
        fig, axes = plt.subplots(n_rows, n_columns, sharex=True, sharey=True,
                                 figsize=(IEEE_COLUMN_WIDTH, height), squeeze=False)
        fig.subplots_adjust(left=left / IEEE_COLUMN_WIDTH, right=1 - right / IEEE_COLUMN_WIDTH,
                            bottom=bottom / height, top=1 - top / height,
                            wspace=col_gap / panel, hspace=row_gap / panel)
        axes = axes.ravel()

        for panel_idx, (ax, model) in enumerate(zip(axes, models)):
            model_df = curves[curves["model_name"] == model]
            ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.6, label="Ideal (y = x)", zorder=1)
            ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.05, color="tomato", linewidth=0)
            ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.05, color="steelblue", linewidth=0)
            # if panel_idx == 0:
            #     region_style = dict(fontsize=6, color="0.35", style="italic", transform=ax.transAxes)
            #     ax.text(0.04, 0.96, "Overconfident\n(error > $\\sigma$)", ha="left", va="top", **region_style)
            #     ax.text(0.96, 0.04, "Underconfident\n(error < $\\sigma$)", ha="right", va="bottom", **region_style)

            for policy in policies:
                policy_curves = model_df.loc[model_df["policy_name"] == policy, "curve"]
                if policy_curves.empty:
                    continue
                policy_curves = np.stack(policy_curves.to_list())
                mean_err = policy_curves.mean(axis=0)
                se_err = policy_curves.std(axis=0) / np.sqrt(len(policy_curves))
                color = POLICY_COLORS[policy]
                ax.plot(sigma_grid, mean_err, color=color, linewidth=1.0, marker=POLICY_MARKERS[policy],
                        markersize=2.5, label=IEEE_POLICY_LABELS[policy], zorder=3)
                ax.fill_between(sigma_grid, mean_err - se_err, mean_err + se_err, color=color, alpha=0.2,
                                linewidth=0, zorder=2)

            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_xticks([0, 0.5, 1])
            ax.set_yticks([0, 0.5, 1])
            ax.set_xticklabels(["0", "0.5", "1"])
            ax.set_yticklabels(["0", "0.5", "1"])
            ax.set_title(MODEL_NAMES[model], pad=2)
            ax.tick_params(pad=1)

        for ax in axes[len(models):]:
            ax.remove()

        fig.supxlabel(r"$\tilde{\sigma}$ (normalized uncertainty per sample)", fontsize=8,
                      y=(bottom - 0.33) / height, va="bottom")
        fig.supylabel("Normalized mean absolute error", fontsize=8, x=0.02 / IEEE_COLUMN_WIDTH, ha="left",
                      y=(bottom + (height - top - bottom) / 2) / height)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False,
                   handlelength=1.5, columnspacing=0.8, handletextpad=0.3, borderaxespad=0.1)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        for ext in ("pdf", "png"):
            fig.savefig(output_dir / f"{filename}.{ext}", dpi=300)
        print(f"Saved: {output_dir / filename}.pdf / .png")
    return fig, axes[:len(models)]


fig, axes = plot_calibration_by_model_ieee(results_df)
plt.show()

In [ ]:
#cover figure (IEEE figure*, 7.16 in): measured nodes -> EDL mean & uncertainty -> MCTS plans the next waypoints
# Uses MODEL_NAMES, POLICY_NAMES and IEEE_RC from the utilities cell.
import matplotlib.patheffects as path_effects
from matplotlib.patches import FancyArrowPatch

IEEE_TEXT_WIDTH = 7.16  # inches, two-column figure*
AGENT_COLORS = ["#4dd0e1", "#ffb74d", "#f06292", "#aed581"]
AGENT_MARKERS = ["o", "s", "^", "D"]


def plot_cover_figure(results_df, model_name="edl", policy_name="mcts", run_id=4, step=16, plan_steps=6,
                      border=8, output_dir=Path("Results/plots"), filename="cover_figure"):
    """
    Teaser figure at one time step of one episode:
    (a) ground truth with the ASV fleet and its past trajectories, (b) measured nodes,
    (c) predicted mean with the ground-truth spill contour, (d) predicted uncertainty with
    the next `plan_steps` waypoints the planner sends each agent to (dashed).
    """
    episode = results_df[(results_df["model_name"] == model_name) & (results_df["policy_name"] == policy_name)
                         & (results_df["run"] == run_id)].sort_values("step").set_index("step")
    row = episode.loc[step]
    crop = (slice(border, -border), slice(border, -border)) if border else (slice(None), slice(None))
    gt = np.asarray(row["ground_truth"], dtype=float)
    mean = np.asarray(row["predicted_mean"], dtype=float)
    sigma = np.asarray(row["predicted_uncertainty"], dtype=float)
    obs_mask = np.asarray(row["obs_mask"]) > 0
    obs_map = np.asarray(row["obs_map"], dtype=float)
    gt_c, mean_c, sigma_c = gt[crop], mean[crop], sigma[crop]

    # Positions (row, col) per step and agent, shifted to the cropped frame
    positions = np.stack([np.asarray(p, dtype=float) for p in episode["position"]]) - border
    steps = episode.index.to_numpy()
    past = positions[steps <= step]
    future = positions[(steps >= step) & (steps <= step + plan_steps)]
    measured_rows, measured_cols = np.nonzero(obs_mask[crop])
    spill = gt_c > 0.05
    coverage = obs_mask[crop].mean()

    # Layout (inches)
    top_header, titles, bottom_loop, side = 0.16, 0.17, 0.36, 0.04
    arrow_gap, pair_gap = 0.34, 0.1
    panel = (IEEE_TEXT_WIDTH - 2 * side - 2 * arrow_gap - pair_gap) / 4
    height = top_header + titles + panel + bottom_loop
    lefts = [side, side + panel + arrow_gap, side + 2 * panel + 2 * arrow_gap,
             side + 3 * panel + 2 * arrow_gap + pair_gap]
    bottom = bottom_loop

    outline = [path_effects.withStroke(linewidth=1.6, foreground="0.1")]

    def fx(x):
        return x / IEEE_TEXT_WIDTH

    def fy(y):
        return y / height

    with plt.rc_context(IEEE_RC):
        fig = plt.figure(figsize=(IEEE_TEXT_WIDTH, height))
        axes = [fig.add_axes([fx(left), fy(bottom), fx(panel), fy(panel)]) for left in lefts]

        def draw_fleet(ax, paths, style, marker_at):
            for agent in range(paths.shape[1]):
                ax.plot(paths[:, agent, 1], paths[:, agent, 0], style, color=AGENT_COLORS[agent],
                        linewidth=0.9, path_effects=outline, zorder=3)
                ax.plot(paths[marker_at, agent, 1], paths[marker_at, agent, 0], AGENT_MARKERS[agent],
                        color=AGENT_COLORS[agent], markersize=4, markeredgecolor="0.1", markeredgewidth=0.6,
                        zorder=4)

        # (a) Ground truth + fleet
        axes[0].imshow(gt_c, cmap="viridis", vmin=0, vmax=1, interpolation="bilinear")
        draw_fleet(axes[0], past, "-", -1)
        # (b) Measured nodes
        axes[1].set_facecolor("0.93")
        axes[1].scatter(measured_cols, measured_rows, c=obs_map[crop][measured_rows, measured_cols], cmap="viridis",
                        vmin=0, vmax=1, s=2.2, marker="s", linewidths=0)
        axes[1].set_xlim(-0.5, gt_c.shape[1] - 0.5)
        axes[1].set_ylim(gt_c.shape[0] - 0.5, -0.5)
        # (c) Predicted mean + ground-truth contour
        im_mean = axes[2].imshow(mean_c, cmap="viridis", vmin=0, vmax=1, interpolation="bilinear")
        axes[2].contour(spill, levels=[0.5], colors="white", linewidths=0.7, linestyles="--")
        # (d) Uncertainty + next waypoints
        im_sigma = axes[3].imshow(sigma_c, cmap="magma", interpolation="bilinear")
        draw_fleet(axes[3], future, "--", 0)
        for agent in range(future.shape[1]):
            axes[3].annotate("", xy=future[-1, agent, ::-1], xytext=future[-2, agent, ::-1],
                             arrowprops=dict(arrowstyle="-|>", color=AGENT_COLORS[agent], lw=0.9,
                                             mutation_scale=6), zorder=4)

        titles_text = [
            "(a) Oil spill & ASV fleet",
            f"(b) Measured nodes ({coverage:.0%} of the map)",
            r"(c) Reconstruction $\mu$",
            r"(d) Uncertainty $\sigma$ & next waypoints",
        ]
        for ax, title in zip(axes, titles_text):
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(0.5)
                spine.set_color("0.3")
            ax.set_title(title, fontsize=7.5, pad=2)
        axes[2].text(0.97, 0.04, f"IoU {row['iou']:.2f}", transform=axes[2].transAxes, ha="right", va="bottom",
                     fontsize=6.5, color="white")

        # Small colorbars inside (c) and (d)
        for ax, im, label in [(axes[2], im_mean, "concentration"), (axes[3], im_sigma, r"$\sigma$")]:
            cax = ax.inset_axes([0.14, 0.06, 0.35, 0.04])
            cbar = fig.colorbar(im, cax=cax, orientation="horizontal")
            cbar.set_ticks([])
            cbar.outline.set_linewidth(0.4)
            cbar.outline.set_edgecolor("white")
            cax.text(-0.04, 0.5, "low", transform=cax.transAxes, ha="right", va="center", fontsize=5.5,
                     color="white")
            cax.text(1.04, 0.5, "high", transform=cax.transAxes, ha="left", va="center", fontsize=5.5,
                     color="white")

        # Pipeline arrows between panels
        mid = fy(bottom + panel / 2)
        for i, label in [(0, "sense"), (1, f"{MODEL_NAMES.get(model_name, model_name)}\nmodel")]:
            x0, x1 = fx(lefts[i] + panel + 0.04), fx(lefts[i + 1] - 0.04)
            fig.patches.append(FancyArrowPatch((x0, mid), (x1, mid), transform=fig.transFigure,
                                               arrowstyle="-|>", mutation_scale=8, lw=1.0, color="0.2"))
            fig.text((x0 + x1) / 2, mid + fy(0.05), label, ha="center", va="bottom", fontsize=6.5,
                     linespacing=1.0)
        # Bracket over (c) and (d): both are outputs of the model
        bracket_y = fy(bottom + panel + titles + 0.02)
        x0, x1 = fx(lefts[2] + 0.05), fx(lefts[3] + panel - 0.05)
        fig.add_artist(plt.Line2D([x0, x0, x1, x1], [bracket_y - fy(0.04), bracket_y, bracket_y,
                                                     bracket_y - fy(0.04)], transform=fig.transFigure,
                                  color="0.3", lw=0.6))
        fig.text((x0 + x1) / 2, bracket_y + fy(0.015), "Uncertainty-aware reconstruction", ha="center",
                 va="bottom", fontsize=7, style="italic")
        # Closed loop: the planner uses sigma to send the fleet to new measurement locations
        y_panels = fy(bottom - 0.03)
        start, end = (fx(lefts[3] + panel / 2), y_panels), (fx(lefts[0] + panel / 2), y_panels)
        fig.patches.append(FancyArrowPatch(start, end, transform=fig.transFigure, arrowstyle="-|>",
                                           mutation_scale=8, lw=1.0, color="0.2",
                                           connectionstyle="bar,fraction=-0.045"))
        loop_y = y_panels - fy(0.045 * (start[0] - end[0]) * IEEE_TEXT_WIDTH)
        fig.text((start[0] + end[0]) / 2, loop_y, f"{POLICY_NAMES.get(policy_name, policy_name)} multi-agent planner: "
                 r"next waypoints driven by $\sigma$", ha="center", va="center", fontsize=7,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor="white", edgecolor="none"))

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        for ext in ("pdf", "png"):
            fig.savefig(output_dir / f"{filename}.{ext}", dpi=300)
        print(f"Saved: {output_dir / filename}.pdf / .png")
    plt.show()
    return fig, axes


fig, axes = plot_cover_figure(results_df)

In [ ]:
#framework cover figure (IEEE figure*, 7.16 in, one row): environment -> surrogate model -> path planning
# Uses MODEL_NAMES, MODEL_COLORS and IEEE_RC (utilities cell) and calibration_curves (calibration cell).
import matplotlib.patheffects as path_effects
from matplotlib.patches import FancyBboxPatch, Polygon

IEEE_TEXT_WIDTH = 7.16  # inches, two-column figure*
AGENT_COLORS = ["#4dd0e1", "#ffb74d", "#f06292", "#aed581"]
BOX_FACE, BOX_EDGE = "#efeeea", "0.6"
MODEL_FACE, MODEL_EDGE = "#f9ece8", "#b5553a"
ENVIRONMENT_FACE, ENVIRONMENT_EDGE = "#eef2fa", "#bfcbe6"
ARROW_COLOR = "0.25"


def _draw_boat(ax, col, row, heading, color, length=10.0):
    """Top view of a catamaran ASV (deck between two pointed hulls) at (col, row), pointing along `heading` (rad)."""
    hull = np.array([[-0.5, -0.09], [0.28, -0.09], [0.5, 0.0], [0.28, 0.09], [-0.5, 0.09]])
    deck = np.array([[-0.3, -0.2], [0.12, -0.2], [0.12, 0.2], [-0.3, 0.2]])
    rotation = np.array([[np.cos(heading), -np.sin(heading)], [np.sin(heading), np.cos(heading)]])
    for shape, offset, face, z in [(deck, 0.0, "0.92", 5), (hull, -0.22, color, 6), (hull, 0.22, color, 6)]:
        local = (shape + [0.0, offset]) * length
        points = local @ rotation.T + [col, row]
        ax.add_patch(Polygon(points, closed=True, facecolor=face, edgecolor="0.1", linewidth=0.4, zorder=z))


def _heading(path, index_from, index_to):
    """Direction (rad, image coordinates) of the segment between two positions of one agent."""
    d_row, d_col = path[index_to] - path[index_from]
    return np.arctan2(d_row, d_col) if (d_row or d_col) else 0.0


def plot_framework_figure(results_df, model_name="edl", policy_name="mcts", run_id=4, step=16, plan_steps=6,
                          border=8, sigma_label=r"\sigma", photo_path=Path("Results/asv_fleet_photo.jpg"),
                          photo_crop=(300, 250, 1140, 880), calibration=None, output_dir=Path("Results/plots"),
                          filename="framework_figure"):
    """
    Compact framework diagram with real data in every block, at one time step of one episode:
    environment (real ASV photo) -> partial observations -> surrogate model (mean and the
    error-uncertainty calibration of every model, highlighting `model_name`, with its boundary
    conditions) -> path planning (uncertainty with the next `plan_steps` waypoints of each agent,
    under constraints) -> new path back to the environment.
    `sigma_label` is the LaTeX symbol of the uncertainty the planner uses; `calibration` is the
    output of calibration_curves(results_df), computed here when not given.
    """
    episode = results_df[(results_df["model_name"] == model_name) & (results_df["policy_name"] == policy_name)
                         & (results_df["run"] == run_id)].sort_values("step").set_index("step")
    row = episode.loc[step]
    crop = (slice(border, -border), slice(border, -border)) if border else (slice(None), slice(None))
    gt = np.asarray(row["ground_truth"], dtype=float)[crop]
    mean = np.asarray(row["predicted_mean"], dtype=float)[crop]
    sigma = np.asarray(row["predicted_uncertainty"], dtype=float)[crop]
    obs_mask = (np.asarray(row["obs_mask"]) > 0)[crop]
    obs_map = np.asarray(row["obs_map"], dtype=float)[crop]
    measured_rows, measured_cols = np.nonzero(obs_mask)

    positions = np.stack([np.asarray(p, dtype=float) for p in episode["position"]]) - border
    steps = episode.index.to_numpy()
    future = positions[(steps >= step) & (steps <= step + plan_steps)]
    n_agents = positions.shape[1]

    # Mean calibration curve of every model over all runs and policies
    sigma_grid, curves = calibration if calibration is not None else calibration_curves(results_df)
    calibration_by_model = {m: np.stack(g["curve"].to_list()).mean(axis=0)
                            for m, g in curves.groupby("model_name", observed=True)}

    x0, y0, x1, y1 = photo_crop
    photo = plt.imread(photo_path)[y0:y1, x0:x1]

    # ── Layout in inches: one row of blocks, side boxes on top, loop below ── #
    W, H, P = IEEE_TEXT_WIDTH, 2.05, 1.0       # figure width/height, map size
    pad, title_h = 0.06, 0.26                  # padding inside blocks, title + subtitle height
    map_y = 0.28
    block_y, block_h = map_y - pad, P + 2 * pad + title_h
    photo_w = P * photo.shape[1] / photo.shape[0]
    cal_gap, cal_w = 0.12, P - 0.02            # room for the calibration y label, calibration plot width
    env_x, env_w, block_w = 0.03, photo_w + 2 * pad, P + 2 * pad
    model_w = 2 * pad + P + cal_gap + cal_w
    gap = (W - 2 * env_x - env_w - P - model_w - block_w) / 3   # same space for every arrow
    obs_x = env_x + env_w + gap
    model_x = obs_x + P + gap
    policy_x = model_x + model_w + gap
    side_y, side_h, side_w = block_y + block_h + 0.12, 0.28, 1.12
    outline = [path_effects.withStroke(linewidth=1.6, foreground="0.1")]

    with plt.rc_context(IEEE_RC):
        fig = plt.figure(figsize=(W, H))
        canvas = fig.add_axes([0, 0, 1, 1], zorder=0)
        canvas.set_xlim(0, W)
        canvas.set_ylim(0, H)
        canvas.axis("off")

        def panel(x, y, w, h):
            ax = fig.add_axes([x / W, y / H, w / W, h / H], zorder=2)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(0.5)
                spine.set_color("0.3")
            return ax

        def box(x, y, w, h, face=BOX_FACE, edge=BOX_EDGE, lw=0.7):
            canvas.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.06",
                                            facecolor=face, edgecolor=edge, linewidth=lw, zorder=1))

        def arrow(points, lw=1.0):
            points = np.asarray(points, dtype=float)
            if len(points) > 2:
                canvas.plot(points[:-1, 0], points[:-1, 1], color=ARROW_COLOR, lw=lw, solid_capstyle="butt",
                            solid_joinstyle="miter", zorder=1.5)
            canvas.annotate("", xy=points[-1], xytext=points[-2], zorder=1.5,
                            arrowprops=dict(arrowstyle="-|>", color=ARROW_COLOR, lw=lw, mutation_scale=7,
                                            shrinkA=0, shrinkB=0))

        def label(x, y, text, **kwargs):
            canvas.text(x, y, text, ha=kwargs.pop("ha", "center"), va=kwargs.pop("va", "center"), zorder=3,
                        **kwargs)

        def block_title(x, w, text, subtitle):
            label(x + w / 2, block_y + block_h - 0.1, text, fontweight="bold", fontsize=7.5)
            label(x + w / 2, block_y + block_h - 0.225, subtitle, fontsize=6.5, style="italic", color="0.25")

        def map_label(ax, text):
            ax.text(0.04, 0.95, text, transform=ax.transAxes, ha="left", va="top", fontsize=6.5, color="white")

        def colorbar(ax, image):
            cax = ax.inset_axes([0.17, 0.06, 0.34, 0.045])
            cbar = fig.colorbar(image, cax=cax, orientation="horizontal")
            cbar.set_ticks([])
            cbar.outline.set_linewidth(0.4)
            cbar.outline.set_edgecolor("white")
            for x, text, ha in [(-0.05, "low", "right"), (1.05, "high", "left")]:
                cax.text(x, 0.5, text, transform=cax.transAxes, ha=ha, va="center", fontsize=5.5, color="white")

        # ── Environment ────────────────────────────────────────────────── #
        box(env_x, block_y, env_w, block_h, ENVIRONMENT_FACE, ENVIRONMENT_EDGE)
        block_title(env_x, env_w, "Environment", "multi-ASV fleet")
        ax_photo = panel(env_x + pad, map_y, photo_w, P)
        ax_photo.imshow(photo)

        # ── Partial observations ───────────────────────────────────────── #
        mid_y = map_y + P / 2
        arrow([(env_x + env_w + 0.03, mid_y), (obs_x - 0.04, mid_y)])
        ax_obs = panel(obs_x, map_y, P, P)
        ax_obs.set_facecolor("0.96")
        ax_obs.scatter(measured_cols, measured_rows, c=obs_map[measured_rows, measured_cols], cmap="viridis",
                       vmin=0, vmax=1, s=1.6, marker="s", linewidths=0)
        ax_obs.set_xlim(-0.5, gt.shape[1] - 0.5)
        ax_obs.set_ylim(gt.shape[0] - 0.5, -0.5)
        block_title(obs_x, P, "Partial observations", "sparse measurements")

        # ── Surrogate model (+ boundary conditions) ─────────────────────── #
        arrow([(obs_x + P + 0.04, mid_y), (model_x - 0.03, mid_y)])
        box(model_x, block_y, model_w, block_h, MODEL_FACE, MODEL_EDGE, lw=0.8)
        block_title(model_x, model_w, "Surrogate model", "evidential deep learning with calibrated uncertainty")
        ax_mean = panel(model_x + pad, map_y, P, P)
        im_mean = ax_mean.imshow(mean, cmap="viridis", vmin=0, vmax=1, interpolation="bilinear")
        ax_mean.contour(gt > 0.05, levels=[0.5], colors="white", linewidths=0.6, linestyles="--")
        map_label(ax_mean, r"$\hat{f}(x)$")
        colorbar(ax_mean, im_mean)

        # Error-uncertainty calibration: the highlighted model follows the diagonal
        ax_cal = fig.add_axes([(model_x + pad + P + cal_gap) / W, (map_y + 0.08) / H, cal_w / W,
                               (P - 0.08) / H], zorder=2)
        ax_cal.fill_between([0, 1], [0, 1], [1, 1], color="tomato", alpha=0.08, linewidth=0, zorder=0)
        ax_cal.fill_between([0, 1], [0, 0], [0, 1], color="steelblue", alpha=0.08, linewidth=0, zorder=0)
        ax_cal.plot([0, 1], [0, 1], "--", color="0.35", linewidth=0.7, zorder=1)
        for other, curve in calibration_by_model.items():
            if other != model_name:
                ax_cal.plot(sigma_grid, curve, color="0.62", linewidth=0.8, alpha=0.9, zorder=2,
                            solid_capstyle="round")
        highlight = MODEL_COLORS[MODEL_NAMES[model_name]]
        ax_cal.plot(sigma_grid, calibration_by_model[model_name], color=highlight, linewidth=1.5, zorder=3,
                    solid_capstyle="round",
                    path_effects=[path_effects.withStroke(linewidth=2.5, foreground="white")])

        region_style = dict(fontsize=5, style="italic", transform=ax_cal.transAxes)
        ax_cal.text(0.04, 0.96, "overconfident", ha="left", va="top", color="#b03a2e", **region_style)
        ax_cal.text(0.96, 0.04, "underconfident", ha="right", va="bottom", color="#2e5e8a", **region_style)
        ax_cal.text(0.42, 0.7, MODEL_NAMES[model_name], ha="center", va="center", fontsize=6.5, color=highlight,
                    fontweight="bold", path_effects=[path_effects.withStroke(linewidth=1.5, foreground="white")])
        ax_cal.text(0.97, 0.36, "other\nmodels", ha="right", va="center", fontsize=5.5, color="0.45",
                    linespacing=0.9)

        ax_cal.set_xlim(0, 1)
        ax_cal.set_ylim(0, 1)
        ax_cal.set_xticks([0, 1])
        ax_cal.set_yticks([0, 1])
        ax_cal.tick_params(labelsize=5.5, length=1.5, pad=1, color="0.4")
        # Axis labels between the 0 and 1 ticks to save space
        ax_cal.set_xlabel(rf"normalized ${sigma_label}$", fontsize=6, labelpad=-6)
        ax_cal.set_ylabel("normalized error", fontsize=6, labelpad=-4)
        for spine in ax_cal.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color("0.4")

        side_x = model_x + (model_w - side_w) / 2
        box(side_x, side_y, side_w, side_h)
        label(side_x + side_w / 2, side_y + side_h / 2, "Boundary conditions", fontweight="bold", fontsize=7)
        arrow([(side_x + side_w / 2, side_y - 0.02), (side_x + side_w / 2, block_y + block_h + 0.02)])

        # ── Surrogate model -> path planning (+ constraints) ─────── #
        arrow([(model_x + model_w + 0.03, mid_y), (policy_x - 0.03, mid_y)])
        gap_x = (model_x + model_w + policy_x) / 2
        label(gap_x, mid_y + 0.13, r"$\hat{f}(x)$", fontsize=7.5)
        label(gap_x, mid_y - 0.12, rf"${sigma_label}$", fontsize=7.5)

        box(policy_x, block_y, block_w, block_h)
        block_title(policy_x, block_w, "Path planning", rf"driven by ${sigma_label}$")
        ax_sigma = panel(policy_x + pad, map_y, P, P)
        im_sigma = ax_sigma.imshow(sigma, cmap="magma", interpolation="bilinear")
        for agent in range(n_agents):
            ax_sigma.plot(future[:, agent, 1], future[:, agent, 0], "--", color=AGENT_COLORS[agent], linewidth=0.9,
                          path_effects=outline, zorder=3)
            ax_sigma.annotate("", xy=future[-1, agent, ::-1], xytext=future[-2, agent, ::-1], zorder=4,
                              arrowprops=dict(arrowstyle="-|>", color=AGENT_COLORS[agent], lw=0.9, mutation_scale=6))
            _draw_boat(ax_sigma, future[0, agent, 1], future[0, agent, 0], _heading(future[:, agent], 0, 1),
                       AGENT_COLORS[agent])
        ax_sigma.set_xlim(-0.5, gt.shape[1] - 0.5)
        ax_sigma.set_ylim(gt.shape[0] - 0.5, -0.5)
        colorbar(ax_sigma, im_sigma)

        side_x = policy_x + (block_w - side_w) / 2
        box(side_x, side_y, side_w, side_h)
        label(side_x + side_w / 2, side_y + side_h / 2, "Constraints", fontweight="bold", fontsize=7)
        arrow([(side_x + side_w / 2, side_y - 0.02), (side_x + side_w / 2, block_y + block_h + 0.02)])

        # ── New path back to the environment ───────────────────────────── #
        loop_y = 0.09
        env_cx, policy_cx = env_x + env_w / 2, policy_x + block_w / 2
        arrow([(policy_cx, block_y - 0.02), (policy_cx, loop_y), (env_cx, loop_y), (env_cx, block_y - 0.02)])
        label((env_cx + policy_cx) / 2, loop_y, "New path", fontsize=7,
              bbox=dict(boxstyle="square,pad=0.2", facecolor="white", edgecolor="none"))

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        for ext in ("pdf", "png"):
            fig.savefig(output_dir / f"{filename}.{ext}", dpi=300)
        print(f"Saved: {output_dir / filename}.pdf / .png")
    plt.show()
    return fig


fig = plot_framework_figure(results_df)